# Credit Risk Scorecard Development & Validation
## Nova Bank, Retail Credit Risk Modelling

This project walks through how I built an end-to-end credit risk scorecard following the methodology used in bank risk modelling teams. The goal was to understand not just the ML part, but the full lifecycle: how scorecards are developed, validated, monitored, and how they feed into Basel III capital and IFRS 9 provisioning.

### What this covers
1. Data quality and cleaning
2. WoE/IV feature selection (the industry standard, not just generic ML feature importance)
3. Logistic regression scorecard (the production model banks actually use)
4. XGBoost + SHAP as a benchmark (to measure the "cost of interpretability")
5. Model validation using Gini, KS, PSI, and calibration (not accuracy/AUC)
6. Basel III IRB capital calculation using the ASRF formula
7. IFRS 9 three-stage ECL classification
8. Stress testing with macro-conditioned PD shocks
9. Early warning framework and fairness assessment
10. A/B model comparison and production monitoring simulation

### References
- [1] Basel Committee (1999). Credit Risk Modelling: Current Practices and Applications.
- [2] Noguer i Alonso & Sun (2025). Credit Risk Modeling for Financial Institutions. SSRN.
- [3] Golec & AlabdulJalil (2025). Interpretable LLMs for Credit Risk. arXiv:2506.04290.
- [4] Hlongwane et al. (2024). Leveraging Shapley values for interpretable credit scorecards. PLoS ONE.


## Setup

In [10]:
!pip install xgboost shap openpyxl -q
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve, classification_report,
    confusion_matrix, precision_recall_curve, brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Try loading SHAP and XGBoost (needed for the ML benchmark section)
try:
    import shap
    import xgboost as xgb
    HAS_SHAP = True
    print("SHAP and XGBoost loaded successfully.")
except ImportError:
    HAS_SHAP = False
    print("SHAP/XGBoost not available. SHAP analysis will be skipped.")

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11, 'figure.dpi': 150,
})
COLORS = {
    'primary': '#1B4F72', 'secondary': '#2E86C1', 'accent': '#E74C3C',
    'good': '#27AE60', 'warn': '#F39C12', 'light': '#AED6F1', 'dark': '#154360',
}
OUTPUT_DIR = 'outputs'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

SHAP and XGBoost loaded successfully.


## 1. Data loading and quality assessment
First things first: load the data, check for issues, and clean up obvious errors before doing anything else.

In [11]:
print("=" * 80)
print("SECTION 1: DATA LOADING & QUALITY ASSESSMENT")
print("=" * 80)

df_raw = pd.read_excel('data/Credit_Risk_Dataset.xlsx')
print(f"\nDataset: {df_raw.shape[0]:,} borrowers × {df_raw.shape[1]} features")
print(f"Default rate: {df_raw['loan_status'].mean():.1%} ({df_raw['loan_status'].sum():,} defaults)")

# Check what is missing
print("\n── Missing Values ──")
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
print(missing_df[missing_df['count'] > 0])

# Look for impossible values that are clearly data entry errors
print(f"\n── Outlier Flags ──")
print(f"  person_age > 100:    {(df_raw['person_age'] > 100).sum()} records (data entry errors)")
print(f"  person_income > $1M: {(df_raw['person_income'] > 1_000_000).sum()} records")
print(f"  person_emp_length > 60 yrs: {(df_raw['person_emp_length'] > 60).sum()} records")

# Remove the obvious errors
df = df_raw.copy()
df = df[df['person_age'] <= 100]
df = df[df['person_income'] <= 1_000_000]
df = df[df['person_emp_length'] <= 60]
print(f"\nAfter cleaning: {len(df):,} records ({len(df_raw) - len(df)} removed)")

# Fill in the gaps: interest rate by loan grade (not global median, since rate depends on grade)
df['person_emp_length'] = df['person_emp_length'].fillna(df['person_emp_length'].median())
df['loan_int_rate'] = df.groupby('loan_grade')['loan_int_rate'].transform(
    lambda x: x.fillna(x.median()))
df['loan_int_rate'] = df['loan_int_rate'].fillna(df['loan_int_rate'].median())

# Create some additional features that might help with risk assessment
df['has_prior_default'] = (df['cb_person_default_on_file'] == 'Y').astype(int)
df['high_utilization'] = (df['credit_utilization_ratio'] > 0.70).astype(int)
df['income_per_age'] = df['person_income'] / df['person_age']
df['total_debt_burden'] = df['loan_amnt'] + df['other_debt']
df['total_burden_to_income'] = df['total_debt_burden'] / df['person_income']

print(f"Remaining missing: {df.isnull().sum().sum()} | Final default rate: {df['loan_status'].mean():.1%}")

SECTION 1: DATA LOADING & QUALITY ASSESSMENT

Dataset: 32,581 borrowers × 29 features
Default rate: 21.8% (7,108 defaults)

── Missing Values ──
                   count   pct
person_emp_length    895  2.75
loan_int_rate       3116  9.56

── Outlier Flags ──
  person_age > 100:    5 records (data entry errors)
  person_income > $1M: 9 records
  person_emp_length > 60 yrs: 2 records

After cleaning: 31,671 records (910 removed)
Remaining missing: 0 | Final default rate: 21.5%


## 2. Exploratory data analysis
Before building any model, I want to see which features show clear differences between defaulters and non-defaulters.

In [12]:
print("\n" + "=" * 80)
print("SECTION 2: EXPLORATORY DATA ANALYSIS")
print("=" * 80)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Default Rate by Key Risk Segments', fontsize=16, fontweight='bold', y=1.02)

cat_vars = [
    ('loan_grade', 'Loan Grade'), ('person_home_ownership', 'Home Ownership'),
    ('loan_intent', 'Loan Purpose'), ('employment_type', 'Employment Type'),
    ('cb_person_default_on_file', 'Prior Default on File'), ('loan_term_months', 'Loan Term'),
]
for idx, (col, title) in enumerate(cat_vars):
    ax = axes[idx // 3, idx % 3]
    grp = df.groupby(col)['loan_status'].agg(['mean', 'count']).reset_index()
    if col == 'loan_grade':
        grp[col] = pd.Categorical(grp[col], categories=['A','B','C','D','E','F','G'], ordered=True)
        grp = grp.sort_values(col)
    else:
        grp = grp.sort_values('mean', ascending=True)
    bars = ax.bar(range(len(grp)), grp['mean'] * 100,
                  color=[COLORS['accent'] if v > 0.30 else COLORS['secondary'] for v in grp['mean']],
                  edgecolor='white')
    ax.set_xticks(range(len(grp)))
    ax.set_xticklabels(grp[col].astype(str), rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Default Rate (%)')
    ax.set_title(title, fontweight='bold')
    ax.axhline(y=df['loan_status'].mean()*100, color=COLORS['dark'], linestyle='--', alpha=0.5)
    for bar, val in zip(bars, grp['mean']):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{val:.1%}', ha='center', va='bottom', fontsize=8, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig1_default_rate_segments.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: fig1_default_rate_segments.png")

# Correlation heatmap to see which features move together
fig, ax = plt.subplots(figsize=(14, 10))
key_numeric = ['loan_status','loan_int_rate','loan_percent_income','debt_to_income_ratio',
               'person_income','loan_amnt','credit_utilization_ratio','person_age',
               'person_emp_length','cb_person_cred_hist_length','past_delinquencies',
               'total_burden_to_income','has_prior_default']
corr = df[key_numeric].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
            square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1, annot_kws={'size': 8})
ax.set_title('Correlation Matrix of Key Risk Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig2_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: fig2_correlation_heatmap.png")


SECTION 2: EXPLORATORY DATA ANALYSIS
Saved: fig1_default_rate_segments.png
Saved: fig2_correlation_heatmap.png


## 3. WoE and Information Value
This is the standard way banks select features for scorecards. Rather than using generic feature importance, WoE bins each feature and measures how well each bin separates good from bad borrowers.

In [13]:
print("\n" + "=" * 80)
print("SECTION 3: WoE & INFORMATION VALUE")
print("=" * 80)
print("""
In banking, the standard way to select features for a credit scorecard is
through WoE (Weight of Evidence) and IV (Information Value). I wanted to
follow this approach rather than just throwing everything into a model,
because this is how practitioners at banks like OCBC actually do it.

The idea: for each feature, bin the values and check how well each bin
separates good borrowers from bad ones.

  WoE = ln(% Non-Defaults in bin / % Defaults in bin)
  IV  = sum of (% Non-Default_i - % Default_i) x WoE_i

IV thresholds (these are standard across the industry):
  < 0.02  Unpredictive
  0.02-0.1 Weak
  0.1-0.3  Medium
  0.3-0.5  Strong
  > 0.5   Suspicious, might be overfitting
""")

def calculate_woe_iv(df, feature, target='loan_status', bins=10):
    temp = df[[feature, target]].copy()
    if temp[feature].dtype in ['object', 'str', 'string', 'category']:
        temp['bin'] = temp[feature]
    else:
        try:
            temp['bin'] = pd.qcut(temp[feature], q=bins, duplicates='drop')
        except:
            temp['bin'] = pd.cut(temp[feature], bins=bins)
    grouped = temp.groupby('bin', observed=True)[target].agg(['sum', 'count'])
    grouped.columns = ['bad', 'total']
    grouped['good'] = grouped['total'] - grouped['bad']
    total_good = grouped['good'].sum()
    total_bad = grouped['bad'].sum()
    grouped['pct_good'] = (grouped['good'] / total_good).replace(0, 0.0001)
    grouped['pct_bad'] = (grouped['bad'] / total_bad).replace(0, 0.0001)
    grouped['woe'] = np.log(grouped['pct_good'] / grouped['pct_bad'])
    grouped['iv'] = (grouped['pct_good'] - grouped['pct_bad']) * grouped['woe']
    return grouped, grouped['iv'].sum()

candidate_features = [
    'person_age','person_income','person_home_ownership','person_emp_length',
    'loan_intent','loan_grade','loan_amnt','loan_int_rate','loan_percent_income',
    'cb_person_default_on_file','cb_person_cred_hist_length','gender','marital_status',
    'education_level','employment_type','loan_term_months','debt_to_income_ratio',
    'credit_utilization_ratio','past_delinquencies','has_prior_default',
    'total_burden_to_income','open_accounts',
]
iv_results = {}
woe_tables = {}
for feat in candidate_features:
    try:
        woe_df, iv_val = calculate_woe_iv(df, feat)
        iv_results[feat] = iv_val
        woe_tables[feat] = woe_df
    except Exception as e:
        pass

iv_df = pd.DataFrame.from_dict(iv_results, orient='index', columns=['IV']).sort_values('IV', ascending=False)
iv_df['Predictive Power'] = pd.cut(iv_df['IV'], bins=[-np.inf,0.02,0.1,0.3,0.5,np.inf],
    labels=['Unpredictive','Weak','Medium','Strong','Suspicious'])
print("\n── Information Value Rankings ──")
print(iv_df.to_string())

# IV chart
fig, ax = plt.subplots(figsize=(12, 8))
colors_iv = ['#8E44AD' if v>=0.5 else COLORS['accent'] if v>=0.3 else COLORS['warn'] if v>=0.1
             else COLORS['secondary'] if v>=0.02 else '#BDC3C7' for v in iv_df['IV']]
bars = ax.barh(range(len(iv_df)), iv_df['IV'], color=colors_iv, edgecolor='white')
ax.set_yticks(range(len(iv_df)))
ax.set_yticklabels(iv_df.index, fontsize=9)
ax.set_xlabel('Information Value')
ax.set_title('Feature Ranking by Information Value (IV)', fontsize=14, fontweight='bold')
ax.axvline(x=0.02, color='gray', linestyle=':', alpha=0.7, label='Unpredictive')
ax.axvline(x=0.1, color='gray', linestyle='--', alpha=0.7, label='Medium')
ax.axvline(x=0.3, color='gray', linestyle='-', alpha=0.7, label='Strong')
ax.legend(fontsize=8, loc='lower right')
ax.invert_yaxis()
for bar, val in zip(bars, iv_df['IV']):
    ax.text(val+0.005, bar.get_y()+bar.get_height()/2, f'{val:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig3_information_value.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: fig3_information_value.png")


SECTION 3: WoE & INFORMATION VALUE

In banking, the standard way to select features for a credit scorecard is
through WoE (Weight of Evidence) and IV (Information Value). I wanted to
follow this approach rather than just throwing everything into a model,
because this is how practitioners at banks like OCBC actually do it.

The idea: for each feature, bin the values and check how well each bin
separates good borrowers from bad ones.

  WoE = ln(% Non-Defaults in bin / % Defaults in bin)
  IV  = sum of (% Non-Default_i - % Default_i) x WoE_i

IV thresholds (these are standard across the industry):
  < 0.02  Unpredictive
  0.02-0.1 Weak
  0.1-0.3  Medium
  0.3-0.5  Strong
  > 0.5   Suspicious, might be overfitting


── Information Value Rankings ──
                                  IV Predictive Power
loan_grade                  0.907561       Suspicious
loan_percent_income         0.879953       Suspicious
loan_int_rate               0.748990       Suspicious
total_burden_to_income     

## 4. Model development
Building two models side by side: (1) logistic regression scorecard as the production candidate, and (2) XGBoost as a benchmark to see how much Gini we are trading for interpretability.

In [14]:
print("\n" + "=" * 80)
print("SECTION 4: MODEL DEVELOPMENT")
print("=" * 80)
print("""
Here I want to build two models side by side and compare them properly.
This is how model governance works in practice: you build multiple
candidates and recommend one with clear rationale.

Banks broadly use two approaches to credit risk modelling:

  (1) Traditional: WoE/IV + logistic regression = points-based scorecard
      This is the industry standard. FICO, Experian, CBS Singapore all
      use this. Banks like OCBC, DBS, UOB use this for production PD
      models. It is fully interpretable, regulatorily accepted, and
      has decades of validation behind it.
      Downside: can miss non-linear patterns.

  (2) ML-based: XGBoost/LightGBM + SHAP/LIME for explainability
      Higher predictive power. Used by fintechs and increasingly
      explored by banks as a performance benchmark.
      Downside: harder to explain, regulators are still skeptical,
      and it tends to be less stable over time.

So I am building both:
  Model A: LR scorecard (the production candidate)
  Model B: XGBoost + SHAP (the benchmark)

The key question: how much Gini are we giving up for interpretability?
If the gap is small, the traditional approach wins easily. If large,
it might be worth exploring hybrid methods like SHAP-based scorecards.
""")

# Pick features that have at least weak predictive power (IV >= 0.02)
selected_feats_iv = iv_df[iv_df['IV'] >= 0.02].index.tolist()

def woe_transform(df_input, feature, target='loan_status', bins=10):
    temp = df_input[[feature, target]].copy()
    if temp[feature].dtype in ['object','str','string','category']:
        bin_col = temp[feature]
    else:
        try: bin_col = pd.qcut(temp[feature], q=bins, duplicates='drop')
        except: bin_col = pd.cut(temp[feature], bins=bins)
    grouped = temp.groupby(bin_col, observed=True)[target].agg(['sum','count'])
    grouped.columns = ['bad','total']
    grouped['good'] = grouped['total'] - grouped['bad']
    total_good = grouped['good'].sum()
    total_bad = grouped['bad'].sum()
    grouped['pct_good'] = (grouped['good']/total_good).replace(0, 0.0001)
    grouped['pct_bad'] = (grouped['bad']/total_bad).replace(0, 0.0001)
    grouped['woe'] = np.log(grouped['pct_good']/grouped['pct_bad'])
    woe_map = grouped['woe'].to_dict()
    return bin_col.map(woe_map), woe_map, bin_col

# Split BEFORE WoE fitting so we don't leak test info into the transformation
X = df.drop(columns=['loan_status','client_ID'])
y = df['loan_status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Train DR: {y_train.mean():.1%} | Test DR: {y_test.mean():.1%}")

train_df = X_train.copy(); train_df['loan_status'] = y_train.values
test_df = X_test.copy(); test_df['loan_status'] = y_test.values

# Drop has_prior_default since cb_person_default_on_file already captures this
redundant = ['has_prior_default']
numeric_feats = [f for f in selected_feats_iv if f in df.columns and df[f].dtype not in ['object','str','string'] and f not in redundant]
cat_feats = [f for f in selected_feats_iv if f in df.columns and df[f].dtype in ['object','str','string']]
all_model_feats = numeric_feats + cat_feats

# WoE transform
woe_maps = {}
X_train_woe = pd.DataFrame(index=X_train.index)
X_test_woe = pd.DataFrame(index=X_test.index)

for feat in all_model_feats:
    try:
        woe_vals, woe_map, bin_col = woe_transform(train_df, feat)
        X_train_woe[f'{feat}_woe'] = woe_vals.values
        woe_maps[feat] = woe_map
        if train_df[feat].dtype in ['object','str','string','category']:
            bin_col_test = test_df[feat]
        else:
            bin_edges = [interval.left for interval in woe_map.keys()] + [list(woe_map.keys())[-1].right]
            try: bin_col_test = pd.cut(test_df[feat], bins=bin_edges, include_lowest=True)
            except: bin_col_test = pd.qcut(test_df[feat], q=10, duplicates='drop')
        X_test_woe[f'{feat}_woe'] = bin_col_test.map(woe_map).fillna(0).values
    except:
        pass

X_train_woe = X_train_woe.dropna(axis=1, how='all')
for col in X_train_woe.columns:
    X_train_woe[col] = pd.to_numeric(X_train_woe[col], errors='coerce')
X_train_woe = X_train_woe.fillna(0)

X_test_woe = X_test_woe.dropna(axis=1, how='all')
for col in X_test_woe.columns:
    X_test_woe[col] = pd.to_numeric(X_test_woe[col], errors='coerce')
X_test_woe = X_test_woe.fillna(0)

common_cols = X_train_woe.columns.intersection(X_test_woe.columns)
X_train_woe = X_train_woe[common_cols]
X_test_woe = X_test_woe[common_cols]

# ── Model A: Logistic Regression ─────────────────────────────────────────────
print("\n── Model A: Logistic Regression Scorecard ──")
lr_model = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs', max_iter=1000, random_state=42)
lr_model.fit(X_train_woe, y_train)
y_train_pred_lr = lr_model.predict_proba(X_train_woe)[:, 1]
y_test_pred_lr = lr_model.predict_proba(X_test_woe)[:, 1]

coef_df = pd.DataFrame({
    'Feature': X_train_woe.columns, 'Coefficient': lr_model.coef_[0],
    'Abs_Coef': np.abs(lr_model.coef_[0])
}).sort_values('Abs_Coef', ascending=False)
print(coef_df.to_string(index=False))

# Points-based scorecard
BASE_SCORE, BASE_ODDS, PDO = 600, 50, 20
FACTOR = PDO / np.log(2)
OFFSET = BASE_SCORE - FACTOR * np.log(BASE_ODDS)
train_scores = OFFSET + FACTOR * np.log((1-y_train_pred_lr)/y_train_pred_lr)
test_scores = OFFSET + FACTOR * np.log((1-y_test_pred_lr)/y_test_pred_lr)
print(f"\nScore range (test): {test_scores.min():.0f}: {test_scores.max():.0f} (mean: {test_scores.mean():.0f})")

# ── Model B: XGBoost with SHAP ───────────────────────────────────────────────
print("\n── Model B: XGBoost Classifier + SHAP Explainability ──")

X_train_ml = X_train.copy()
X_test_ml = X_test.copy()
drop_cols = ['city','state','city_latitude','city_longitude','country']
X_train_ml = X_train_ml.drop(columns=[c for c in drop_cols if c in X_train_ml.columns], errors='ignore')
X_test_ml = X_test_ml.drop(columns=[c for c in drop_cols if c in X_test_ml.columns], errors='ignore')

le_dict = {}
for col in X_train_ml.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X_train_ml[col] = le.fit_transform(X_train_ml[col].astype(str))
    X_test_ml[col] = X_test_ml[col].astype(str).map(
        dict(zip(le.classes_, le.transform(le.classes_)))).fillna(-1).astype(int)
    le_dict[col] = le
X_train_ml = X_train_ml.fillna(0)
X_test_ml = X_test_ml.fillna(0)

if HAS_SHAP:
    xgb_model = xgb.XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.1, subsample=0.8,
        min_child_weight=50, reg_alpha=0.1, reg_lambda=1.0,
        use_label_encoder=False, eval_metric='logloss', random_state=42
    )
    xgb_model.fit(X_train_ml, y_train)
    y_train_pred_gb = xgb_model.predict_proba(X_train_ml)[:, 1]
    y_test_pred_gb = xgb_model.predict_proba(X_test_ml)[:, 1]
else:
    gb_model = GradientBoostingClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.1, subsample=0.8,
        min_samples_leaf=50, random_state=42)
    gb_model.fit(X_train_ml, y_train)
    y_train_pred_gb = gb_model.predict_proba(X_train_ml)[:, 1]
    y_test_pred_gb = gb_model.predict_proba(X_test_ml)[:, 1]


SECTION 4: MODEL DEVELOPMENT

Here I want to build two models side by side and compare them properly.
This is how model governance works in practice: you build multiple
candidates and recommend one with clear rationale.

Banks broadly use two approaches to credit risk modelling:

  (1) Traditional: WoE/IV + logistic regression = points-based scorecard
      This is the industry standard. FICO, Experian, CBS Singapore all
      use this. Banks like OCBC, DBS, UOB use this for production PD
      models. It is fully interpretable, regulatorily accepted, and
      has decades of validation behind it.
      Downside: can miss non-linear patterns.

  (2) ML-based: XGBoost/LightGBM + SHAP/LIME for explainability
      Higher predictive power. Used by fintechs and increasingly
      explored by banks as a performance benchmark.
      Downside: harder to explain, regulators are still skeptical,
      and it tends to be less stable over time.

So I am building both:
  Model A: LR scorecard (th

## 5. SHAP explainability
Using SHAP to make the XGBoost model interpretable. This also lets me cross-validate the WoE feature selection: do SHAP and IV agree on what matters?

In [15]:
print("\n" + "=" * 80)
print("SECTION 5: SHAP EXPLAINABILITY (Explainable AI)")
print("=" * 80)
print("""
SHAP (SHapley Additive exPlanations) is one of the things I picked up from
reading Golec & AlabdulJalil (2025). They reviewed 60 papers on LLMs and ML
in credit risk, and SHAP came up as the most widely used explainability
technique in the field right now.

What SHAP gives us:
  Global view: which features matter most across all borrowers
  Local view: why THIS specific borrower was flagged as risky

This is useful because XGBoost outperforms logistic regression on Gini,
but without SHAP it would be a black box. SHAP makes it auditable.

There is also recent work by Hlongwane et al. (2024) showing that
SHAP-based scorecards from tree models can produce credit scores
comparable to logistic regression while keeping the better accuracy.
That is an interesting direction for the future.
""")

if HAS_SHAP:
    # Global SHAP: which features matter most across the whole portfolio
    print("Computing SHAP values (this may take a moment)...")
    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_test_ml)

    # Global feature importance: SHAP summary bar plot
    fig, ax = plt.subplots(figsize=(12, 8))
    shap.summary_plot(shap_values, X_test_ml, plot_type="bar", show=False, max_display=15)
    plt.title('SHAP Global Feature Importance (XGBoost)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/fig4_shap_global_importance.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: fig4_shap_global_importance.png")

    # SHAP beeswarm: shows direction of impact
    fig, ax = plt.subplots(figsize=(12, 10))
    shap.summary_plot(shap_values, X_test_ml, show=False, max_display=15)
    plt.title('SHAP Beeswarm Plot: Feature Impact Direction', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/fig5_shap_beeswarm.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: fig5_shap_beeswarm.png")

    # Local explanation: pick one high-risk borrower and show why they were flagged
    # Pick a high-risk borrower from test set
    high_risk_idx = np.argmax(y_test_pred_gb)
    fig, ax = plt.subplots(figsize=(14, 4))
    shap.plots.waterfall(shap.Explanation(
        values=shap_values[high_risk_idx],
        base_values=explainer.expected_value,
        data=X_test_ml.iloc[high_risk_idx],
        feature_names=X_test_ml.columns.tolist()
    ), show=False, max_display=12)
    plt.title(f'SHAP Local Explanation: High-Risk Borrower (PD={y_test_pred_gb[high_risk_idx]:.1%})',
              fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/fig6_shap_local_explanation.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: fig6_shap_local_explanation.png")

    # Interesting check: do SHAP and IV agree on which features matter?
    shap_importance = pd.DataFrame({
        'Feature': X_test_ml.columns,
        'Mean_SHAP': np.abs(shap_values).mean(axis=0)
    }).sort_values('Mean_SHAP', ascending=False)
    print("\n── SHAP vs IV Feature Ranking Comparison ──")
    print("Top 10 by SHAP:")
    print(shap_importance.head(10).to_string(index=False))
    print("\nTop 10 by IV:")
    print(iv_df.head(10).to_string())
    print("""
Note: SHAP and IV rankings largely agree on top features (loan grade,
loan-to-income, interest rate), which validates our WoE scorecard's
feature selection. Where they diverge, SHAP captures non-linear
interactions that WoE (univariate by nature) misses.
""")
else:
    print("SHAP not available: skipping XAI analysis.")
    gb_model_for_imp = gb_model if not HAS_SHAP else None


SECTION 5: SHAP EXPLAINABILITY (Explainable AI)

SHAP (SHapley Additive exPlanations) is one of the things I picked up from
reading Golec & AlabdulJalil (2025). They reviewed 60 papers on LLMs and ML
in credit risk, and SHAP came up as the most widely used explainability
technique in the field right now.

What SHAP gives us:
  Global view: which features matter most across all borrowers
  Local view: why THIS specific borrower was flagged as risky

This is useful because XGBoost outperforms logistic regression on Gini,
but without SHAP it would be a black box. SHAP makes it auditable.

There is also recent work by Hlongwane et al. (2024) showing that
SHAP-based scorecards from tree models can produce credit scores
comparable to logistic regression while keeping the better accuracy.
That is an interesting direction for the future.

Computing SHAP values (this may take a moment)...
Saved: fig4_shap_global_importance.png
Saved: fig5_shap_beeswarm.png
Saved: fig6_shap_local_explanation.pn

## 6. Model validation
Banks use Gini, KS, PSI, and calibration curves. Not accuracy, not AUC. This section evaluates both models using the metrics that matter in credit risk.

In [16]:
print("\n" + "=" * 80)
print("SECTION 6: MODEL VALIDATION")
print("=" * 80)
print("""
One thing I learned from the Basel Committee (1999) paper is that credit risk
model validation is fundamentally harder than market risk. Market risk models
can be backtested daily because you get new price data every day. Credit risk
models use a 1-year horizon and defaults are rare events, so you need years
of data spanning multiple economic cycles to validate properly.

Banks use specific metrics for this (not just accuracy or AUC):
  Gini coefficient: 2 x AUC - 1, measures discrimination power
  KS statistic: max separation between cumulative good/bad distributions
  CAP curve and accuracy ratio
  PSI: Population Stability Index, detects if the population has shifted
  Calibration: checks if predicted PD actually matches observed default rate
""")

def gini_coefficient(y_true, y_pred):
    return 2 * roc_auc_score(y_true, y_pred) - 1

def ks_statistic(y_true, y_pred):
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    return np.max(tpr - fpr)

def calculate_psi(expected, actual, bins=10):
    breakpoints = np.unique(np.percentile(expected, np.linspace(0, 100, bins + 1)))
    exp_counts = np.histogram(expected, bins=breakpoints)[0]
    act_counts = np.histogram(actual, bins=breakpoints)[0]
    exp_pct = np.where(exp_counts == 0, 0.0001, exp_counts / exp_counts.sum())
    act_pct = np.where(act_counts == 0, 0.0001, act_counts / act_counts.sum())
    return np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct))

metrics = {}
for name, y_tr_p, y_te_p in [('Logistic Regression', y_train_pred_lr, y_test_pred_lr),
                               ('XGBoost', y_train_pred_gb, y_test_pred_gb)]:
    m = {}
    m['AUC_train'] = roc_auc_score(y_train, y_tr_p)
    m['AUC_test'] = roc_auc_score(y_test, y_te_p)
    m['Gini_train'] = gini_coefficient(y_train, y_tr_p)
    m['Gini_test'] = gini_coefficient(y_test, y_te_p)
    m['KS_train'] = ks_statistic(y_train, y_tr_p)
    m['KS_test'] = ks_statistic(y_test, y_te_p)
    m['PSI'] = calculate_psi(y_tr_p, y_te_p)
    m['Brier_test'] = brier_score_loss(y_test, y_te_p)
    metrics[name] = m

metrics_df = pd.DataFrame(metrics).T
print("\n── Model Performance Summary ──")
print(metrics_df.round(4).to_string())

for name in metrics:
    gap = metrics[name]['Gini_train'] - metrics[name]['Gini_test']
    print(f"\n{name}: Gini gap: {gap:.4f}", end="")
    print(" ⚠️  Overfitting risk" if gap > 0.05 else " ✓ Stable")

# Validation plots
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# ROC
ax = axes[0]
for name, y_pred, color, ls in [
    ('LR (Test)', y_test_pred_lr, COLORS['primary'], '-'),
    ('XGB (Test)', y_test_pred_gb, COLORS['accent'], '-'),
]:
    fpr, tpr, _ = roc_curve(y_test, y_pred)
    auc_val = roc_auc_score(y_test, y_pred)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})', color=color, linestyle=ls, linewidth=2)
ax.plot([0,1],[0,1],'k--',alpha=0.3)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curve', fontweight='bold'); ax.legend(fontsize=9)

# KS
ax = axes[1]
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_test_pred_lr)
ks_val = np.max(tpr_lr - fpr_lr)
ks_idx = np.argmax(tpr_lr - fpr_lr)
ax.plot(np.linspace(0,1,len(tpr_lr)), tpr_lr, label='Cum Bad (TPR)', color=COLORS['accent'], linewidth=2)
ax.plot(np.linspace(0,1,len(fpr_lr)), fpr_lr, label='Cum Good (FPR)', color=COLORS['good'], linewidth=2)
ax.annotate(f'KS = {ks_val:.3f}', xy=(np.linspace(0,1,len(tpr_lr))[ks_idx], (tpr_lr[ks_idx]+fpr_lr[ks_idx])/2),
            fontsize=12, fontweight='bold', xytext=(20,0), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color=COLORS['dark']))
ax.set_title('KS Plot, LR Scorecard (Test Set)', fontweight='bold'); ax.legend(fontsize=9)

# Calibration
ax = axes[2]
for name, y_pred, color in [('LR', y_test_pred_lr, COLORS['primary']), ('XGB', y_test_pred_gb, COLORS['accent'])]:
    prob_true, prob_pred = calibration_curve(y_test, y_pred, n_bins=10, strategy='quantile')
    ax.plot(prob_pred, prob_true, 's-', label=name, color=color, linewidth=2)
ax.plot([0,1],[0,1],'k--',alpha=0.3)
ax.set_xlabel('Predicted PD'); ax.set_ylabel('Observed DR')
ax.set_title('Calibration Curve', fontweight='bold'); ax.legend(fontsize=9)

plt.suptitle('Model Validation Metrics', fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig7_model_validation.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: fig7_model_validation.png")


SECTION 6: MODEL VALIDATION

One thing I learned from the Basel Committee (1999) paper is that credit risk
model validation is fundamentally harder than market risk. Market risk models
can be backtested daily because you get new price data every day. Credit risk
models use a 1-year horizon and defaults are rare events, so you need years
of data spanning multiple economic cycles to validate properly.

Banks use specific metrics for this (not just accuracy or AUC):
  Gini coefficient: 2 x AUC - 1, measures discrimination power
  KS statistic: max separation between cumulative good/bad distributions
  CAP curve and accuracy ratio
  PSI: Population Stability Index, detects if the population has shifted
  Calibration: checks if predicted PD actually matches observed default rate


── Model Performance Summary ──
                     AUC_train  AUC_test  Gini_train  Gini_test  KS_train  KS_test     PSI  Brier_test
Logistic Regression     0.8784    0.8685      0.7567     0.7370    0.6161   0

## 7. Basel III IRB capital (ASRF framework)
Implementing the actual regulatory capital formula. The ASRF model asks: if the economy hits its worst 0.1% scenario, how much capital does the bank need?

In [17]:
print("\n" + "=" * 80)
print("SECTION 7: BASEL III IRB REGULATORY CAPITAL (ASRF Framework)")
print("=" * 80)
print("""
This section implements the actual Basel III IRB capital formula, which I
studied from Noguer i Alonso & Sun (2025). The formula is based on the
ASRF (Asymptotic Single Risk Factor) model.

The core idea: every borrower's financial health depends on two things:
  (1) The economy (a systematic factor Y, shared by everyone)
  (2) Their own personal circumstances (an idiosyncratic factor epsilon)

  Z_i = sqrt(rho) x Y + sqrt(1-rho) x epsilon_i

If the economy crashes (Y at the 0.1th percentile), what happens to the
borrower's PD? The formula computes this "stressed PD":

  PD_stressed = Phi( [Phi_inv(PD) + sqrt(rho) x Phi_inv(0.999)] / sqrt(1-rho) )

Capital requirement = LGD x (PD_stressed - PD)

The capital covers the gap between what you expect to lose on average
and what you could lose in a once-in-1000-years economic crash.
""")

from scipy.stats import norm

def basel_irb_capital(pd_val, lgd=0.45, maturity=2.5, ead=1.0):
    """
    Basel III IRB capital formula for retail exposures.
    For retail: asset correlation ρ = 0.03×(1-exp(-35×PD))/(1-exp(-35)) + 0.16×(1-(1-exp(-35×PD))/(1-exp(-35)))
    Simplified: ρ ranges from 0.03 to 0.16 for retail.
    Maturity adjustment applies to non-retail; simplified here.
    """
    # Asset correlation for "other retail" (Basel III)
    rho = 0.03 * (1 - np.exp(-35 * pd_val)) / (1 - np.exp(-35)) + \
          0.16 * (1 - (1 - np.exp(-35 * pd_val)) / (1 - np.exp(-35)))

    # Conditional PD at 99.9% confidence
    pd_stressed = norm.ppf(
        (norm.ppf(pd_val) + np.sqrt(rho) * norm.ppf(0.999)) / np.sqrt(1 - rho)
    )
    # Clip to valid range
    pd_stressed = np.clip(pd_stressed, 0, 1)

    # Maturity adjustment (simplified for retail = 1.0)
    # For non-retail: b = (0.11852 - 0.05478*ln(PD))^2, MA = (1+(M-2.5)*b)/(1-1.5*b)
    ma = 1.0  # retail

    # Capital requirement (K)
    k = lgd * (pd_stressed - pd_val) * ma
    k = np.maximum(k, 0)

    # Risk-Weighted Assets
    rwa = k * 12.5 * ead

    return pd_stressed, rho, k, rwa

# Apply to test set
test_capital = pd.DataFrame({
    'predicted_pd': y_test_pred_lr,
    'loan_amnt': X_test['loan_amnt'].values,
    'actual_default': y_test.values,
})

# Compute capital for each borrower
results = test_capital['predicted_pd'].apply(
    lambda pd_val: basel_irb_capital(max(pd_val, 0.0003), lgd=0.45)  # Floor PD at 3bps per Basel
)
test_capital['pd_stressed'] = [r[0] for r in results]
test_capital['asset_corr'] = [r[1] for r in results]
test_capital['capital_k'] = [r[2] for r in results]
test_capital['rwa'] = [r[3] for r in results]
test_capital['capital_amount'] = test_capital['capital_k'] * test_capital['loan_amnt']
test_capital['rwa_amount'] = test_capital['rwa'] * test_capital['loan_amnt']

# Score bands for capital analysis
test_capital['score'] = test_scores
test_capital['score_band'] = pd.qcut(test_capital['score'], q=10, duplicates='drop')

cap_by_band = test_capital.groupby('score_band', observed=True).agg(
    n_loans=('predicted_pd', 'count'),
    avg_pd=('predicted_pd', 'mean'),
    avg_pd_stressed=('pd_stressed', 'mean'),
    avg_asset_corr=('asset_corr', 'mean'),
    total_exposure=('loan_amnt', 'sum'),
    total_capital=('capital_amount', 'sum'),
    total_rwa=('rwa_amount', 'sum'),
).reset_index()
cap_by_band['capital_pct'] = (cap_by_band['total_capital'] / cap_by_band['total_exposure'] * 100).round(2)
cap_by_band['rwa_density'] = (cap_by_band['total_rwa'] / cap_by_band['total_exposure'] * 100).round(1)

print("\n── Basel IRB Capital by Score Band ──")
print(cap_by_band[['score_band','n_loans','avg_pd','avg_pd_stressed','avg_asset_corr',
                     'capital_pct','rwa_density']].to_string(index=False))

print(f"\n── Portfolio Summary ──")
print(f"Total Exposure: ${test_capital['loan_amnt'].sum():,.0f}")
print(f"Total RWA:      ${test_capital['rwa_amount'].sum():,.0f}")
print(f"Total Capital:  ${test_capital['capital_amount'].sum():,.0f}")
print(f"Avg Capital %:  {test_capital['capital_amount'].sum()/test_capital['loan_amnt'].sum()*100:.2f}%")
print(f"Avg RWA Density: {test_capital['rwa_amount'].sum()/test_capital['loan_amnt'].sum()*100:.1f}%")

# Basel capital chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
x = range(len(cap_by_band))
ax.bar(x, cap_by_band['avg_pd']*100, width=0.4, label='PD (PIT)', color=COLORS['secondary'], alpha=0.8)
ax.bar([i+0.4 for i in x], cap_by_band['avg_pd_stressed']*100, width=0.4,
       label='PD Stressed (99.9%)', color=COLORS['accent'], alpha=0.8)
ax.set_ylabel('Probability of Default (%)')
ax.set_title('PIT PD vs Basel Stressed PD by Score Band', fontweight='bold')
ax.set_xticks([i+0.2 for i in x])
ax.set_xticklabels([f'Band {i+1}' for i in x], rotation=45, fontsize=8)
ax.legend()

ax = axes[1]
bars = ax.bar(x, cap_by_band['capital_pct'],
    color=[COLORS['good'] if v<5 else COLORS['warn'] if v<15 else COLORS['accent'] for v in cap_by_band['capital_pct']],
    edgecolor='white')
ax.set_ylabel('Capital Requirement (% of Exposure)')
ax.set_title('Basel IRB Capital Requirement by Score Band', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'Band {i+1}' for i in x], rotation=45, fontsize=8)
for bar, val in zip(bars, cap_by_band['capital_pct']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{val:.1f}%', ha='center', fontsize=8, fontweight='bold')

plt.suptitle('Basel III IRB Capital Analysis (ASRF Framework)', fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig8_basel_irb_capital.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: fig8_basel_irb_capital.png")


SECTION 7: BASEL III IRB REGULATORY CAPITAL (ASRF Framework)

This section implements the actual Basel III IRB capital formula, which I
studied from Noguer i Alonso & Sun (2025). The formula is based on the
ASRF (Asymptotic Single Risk Factor) model.

The core idea: every borrower's financial health depends on two things:
  (1) The economy (a systematic factor Y, shared by everyone)
  (2) Their own personal circumstances (an idiosyncratic factor epsilon)

  Z_i = sqrt(rho) x Y + sqrt(1-rho) x epsilon_i

If the economy crashes (Y at the 0.1th percentile), what happens to the
borrower's PD? The formula computes this "stressed PD":

  PD_stressed = Phi( [Phi_inv(PD) + sqrt(rho) x Phi_inv(0.999)] / sqrt(1-rho) )

Capital requirement = LGD x (PD_stressed - PD)

The capital covers the gap between what you expect to lose on average
and what you could lose in a once-in-1000-years economic crash.


── Basel IRB Capital by Score Band ──
        score_band  n_loans   avg_pd  avg_pd_stressed  avg

## 8. IFRS 9 stage classification and ECL
IFRS 9 requires forward-looking provisioning. Loans are classified into three stages, and the jump from Stage 1 to Stage 2 can triple the provision even if the borrower has not missed a payment.

In [18]:
print("\n" + "=" * 80)
print("SECTION 8: IFRS 9 STAGE CLASSIFICATION & ECL")
print("=" * 80)
print("""
IFRS 9 is the accounting standard that tells banks how much to provision
for expected loan losses. Unlike Basel (which is about capital), IFRS 9
directly affects the bank's reported profit.

The key concept is the three-stage model:
  Stage 1: Loan is performing fine. Provision for 12-month ECL only.
  Stage 2: Significant Increase in Credit Risk (SICR) detected, but no
           default yet. Provision jumps to lifetime ECL.
  Stage 3: Borrower has defaulted. Provision = LGD x EAD.

Important: IFRS 9 uses Point-in-Time PD (reflects current conditions),
not Through-the-Cycle PD (which Basel uses for capital).

For this analysis:
  LGD = 45% (Basel Foundation IRB assumption for unsecured retail)
  Lifetime PD = 1 - (1 - PD_annual)^T where T = remaining loan term
""")

LGD = 0.45

test_ifrs = pd.DataFrame({
    'predicted_pd': y_test_pred_lr,
    'actual_default': y_test.values,
    'loan_amnt': X_test['loan_amnt'].values,
    'dti': X_test['debt_to_income_ratio'].values,
    'past_delinq': X_test['past_delinquencies'].values,
    'prior_default': (X_test['cb_person_default_on_file'] == 'Y').values,
    'credit_util': X_test['credit_utilization_ratio'].values,
    'loan_term': X_test['loan_term_months'].values,
})

def classify_ifrs9(row):
    if row['actual_default'] == 1:
        return 'Stage 3'
    triggers = sum([
        row['predicted_pd'] > 0.30,
        row['past_delinq'] > 0,
        row['prior_default'],
        row['dti'] > 0.50,
        row['credit_util'] > 0.80,
    ])
    return 'Stage 2' if triggers >= 2 else 'Stage 1'

test_ifrs['stage'] = test_ifrs.apply(classify_ifrs9, axis=1)

def calc_ecl(row):
    pd_val = row['predicted_pd']
    ead = row['loan_amnt']
    if row['stage'] == 'Stage 1':
        return pd_val * LGD * ead
    elif row['stage'] == 'Stage 2':
        term_years = row['loan_term'] / 12
        lifetime_pd = min(1 - (1 - pd_val) ** term_years, 1.0)
        return lifetime_pd * LGD * ead
    else:
        return LGD * ead

test_ifrs['ecl'] = test_ifrs.apply(calc_ecl, axis=1)

stage_summary = test_ifrs.groupby('stage').agg(
    count=('actual_default','count'), n_defaults=('actual_default','sum'),
    default_rate=('actual_default','mean'), avg_pd=('predicted_pd','mean'),
    total_exposure=('loan_amnt','sum'), total_ecl=('ecl','sum'),
).reset_index()
stage_summary['pct_portfolio'] = (stage_summary['count']/stage_summary['count'].sum()*100).round(1)
stage_summary['ecl_rate'] = (stage_summary['total_ecl']/stage_summary['total_exposure']*100).round(2)

print("\n── IFRS 9 Stage Summary ──")
print(stage_summary.to_string(index=False))
print(f"\nTotal ECL: ${stage_summary['total_ecl'].sum():,.0f}")
print(f"ECL Rate: {stage_summary['total_ecl'].sum()/stage_summary['total_exposure'].sum():.2%}")

# IFRS 9 chart
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
stage_colors = {'Stage 1': COLORS['good'], 'Stage 2': COLORS['warn'], 'Stage 3': COLORS['accent']}

for i, (metric, ylabel, title) in enumerate([
    ('count', 'Number of Loans', 'Loan Count by IFRS 9 Stage'),
    ('default_rate', 'Default Rate', 'Default Rate by Stage'),
    ('ecl_rate', 'ECL Rate (%)', 'ECL Coverage Rate by Stage'),
]):
    ax = axes[i]
    vals = stage_summary[metric] * (100 if metric == 'default_rate' else 1)
    bars = ax.bar(stage_summary['stage'], vals,
                  color=[stage_colors[s] for s in stage_summary['stage']], edgecolor='white')
    ax.set_ylabel(ylabel); ax.set_title(title, fontweight='bold')
    for bar, v in zip(bars, stage_summary[metric]):
        fmt = f'{v:.1%}' if metric == 'default_rate' else f'{v:,.0f}' if metric == 'count' else f'{v:.1f}%'
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02, fmt,
                ha='center', fontweight='bold', fontsize=10)

plt.suptitle('IFRS 9 Expected Credit Loss Analysis', fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig9_ifrs9_staging.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: fig9_ifrs9_staging.png")


SECTION 8: IFRS 9 STAGE CLASSIFICATION & ECL

IFRS 9 is the accounting standard that tells banks how much to provision
for expected loan losses. Unlike Basel (which is about capital), IFRS 9
directly affects the bank's reported profit.

The key concept is the three-stage model:
  Stage 1: Loan is performing fine. Provision for 12-month ECL only.
  Stage 2: Significant Increase in Credit Risk (SICR) detected, but no
           default yet. Provision jumps to lifetime ECL.
  Stage 3: Borrower has defaulted. Provision = LGD x EAD.

Important: IFRS 9 uses Point-in-Time PD (reflects current conditions),
not Through-the-Cycle PD (which Basel uses for capital).

For this analysis:
  LGD = 45% (Basel Foundation IRB assumption for unsecured retail)
  Lifetime PD = 1 - (1 - PD_annual)^T where T = remaining loan term


── IFRS 9 Stage Summary ──
  stage  count  n_defaults  default_rate   avg_pd  total_exposure    total_ecl  pct_portfolio  ecl_rate
Stage 1   6000           0           0.0 0.08925

## 9. Stress testing
What happens to the portfolio if the economy crashes? This section applies macro-conditioned PD shocks across four scenarios.

In [19]:
print("\n" + "=" * 80)
print("SECTION 9: STRESS TESTING (PIT vs TTC)")
print("=" * 80)
print("""
The scorecard gives us PDs under current conditions. But what if the economy
tanks? Stress testing answers that question by applying macro-conditioned
shocks to the PDs and seeing how the portfolio holds up.

I am running four scenarios:
  Baseline:       current PDs as-is
  Mild downturn:  PDs increase 50% (like a GDP drop of 1%)
  Severe stress:  PDs increase 100% (GDP -3%, unemployment +4%)
  Systemic crisis: PDs increase 200% (think 2008-level shock)

This connects to the Basel Committee (1999) paper's discussion of
conditional vs unconditional models. Our scorecard is unconditional (it
does not explicitly factor in the macro state). Stress testing makes it
conditional by saying: given THIS economy, what do the PDs become?
""")

scenarios = {
    'Baseline': 1.0,
    'Mild Downturn (GDP -1%)': 1.5,
    'Severe Stress (GDP -3%)': 2.0,
    'Systemic Crisis': 3.0,
}

stress_results = []
base_pds = y_test_pred_lr.copy()

for scenario_name, pd_multiplier in scenarios.items():
    stressed_pds = np.clip(base_pds * pd_multiplier, 0, 0.999)

    # Recalculate ECL under stressed PDs
    total_ecl = 0
    total_capital = 0
    total_exposure = X_test['loan_amnt'].values.sum()

    for i in range(len(stressed_pds)):
        pd_val = max(stressed_pds[i], 0.0003)
        ead = X_test['loan_amnt'].values[i]

        # ECL (simplified: use stressed PD for all stages)
        total_ecl += pd_val * LGD * ead

        # Basel capital
        _, _, k, _ = basel_irb_capital(pd_val, lgd=LGD)
        total_capital += k * ead

    # Portfolio default rate under stress
    stressed_default_rate = stressed_pds.mean()

    stress_results.append({
        'Scenario': scenario_name,
        'PD Multiplier': f'{pd_multiplier:.1f}x',
        'Avg PD': f'{stressed_default_rate:.1%}',
        'Total ECL ($M)': f'{total_ecl/1e6:.2f}',
        'ECL Rate': f'{total_ecl/total_exposure:.2%}',
        'Capital ($M)': f'{total_capital/1e6:.2f}',
        'Capital Rate': f'{total_capital/total_exposure:.2%}',
    })

stress_df = pd.DataFrame(stress_results)
print("\n── Stress Test Results ──")
print(stress_df.to_string(index=False))

# Stress test chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

scenario_names = [s['Scenario'] for s in stress_results]
ecl_values = [float(s['Total ECL ($M)']) for s in stress_results]
capital_values = [float(s['Capital ($M)']) for s in stress_results]

ax = axes[0]
colors_stress = [COLORS['good'], COLORS['warn'], COLORS['accent'], '#8E44AD']
bars = ax.bar(range(len(scenario_names)), ecl_values, color=colors_stress, edgecolor='white')
ax.set_xticks(range(len(scenario_names)))
ax.set_xticklabels(scenario_names, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('ECL ($M)')
ax.set_title('Expected Credit Loss Under Stress Scenarios', fontweight='bold')
for bar, val in zip(bars, ecl_values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1, f'${val:.1f}M',
            ha='center', fontsize=9, fontweight='bold')

ax = axes[1]
bars = ax.bar(range(len(scenario_names)), capital_values, color=colors_stress, edgecolor='white')
ax.set_xticks(range(len(scenario_names)))
ax.set_xticklabels(scenario_names, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Required Capital ($M)')
ax.set_title('Basel IRB Capital Under Stress Scenarios', fontweight='bold')
for bar, val in zip(bars, capital_values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1, f'${val:.1f}M',
            ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Macro-Conditioned Stress Testing', fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig10_stress_testing.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: fig10_stress_testing.png")


SECTION 9: STRESS TESTING (PIT vs TTC)

The scorecard gives us PDs under current conditions. But what if the economy
tanks? Stress testing answers that question by applying macro-conditioned
shocks to the PDs and seeing how the portfolio holds up.

I am running four scenarios:
  Baseline:       current PDs as-is
  Mild downturn:  PDs increase 50% (like a GDP drop of 1%)
  Severe stress:  PDs increase 100% (GDP -3%, unemployment +4%)
  Systemic crisis: PDs increase 200% (think 2008-level shock)

This connects to the Basel Committee (1999) paper's discussion of
conditional vs unconditional models. Our scorecard is unconditional (it
does not explicitly factor in the macro state). Stress testing makes it
conditional by saying: given THIS economy, what do the PDs become?


── Stress Test Results ──
               Scenario PD Multiplier Avg PD Total ECL ($M) ECL Rate Capital ($M) Capital Rate
               Baseline          1.0x  20.7%          10.26   11.06%          nan         nan%
Mild

## 10. Early warning and fairness assessment
Identifying borrowers with multiple risk flags before default, and verifying that protected attributes (gender, education, country) have no predictive power.

In [20]:
print("\n" + "=" * 80)
print("SECTION 10: EARLY WARNING & FAIRNESS ASSESSMENT")
print("=" * 80)

# Red flag system
ew_df = df.copy()
ew_df['high_dti'] = (ew_df['debt_to_income_ratio'] > 0.40).astype(int)
ew_df['high_lpi'] = (ew_df['loan_percent_income'] > 0.30).astype(int)
ew_df['high_rate'] = (ew_df['loan_int_rate'] > 15).astype(int)
ew_df['renter'] = (ew_df['person_home_ownership'] == 'RENT').astype(int)
ew_df['prior_default'] = (ew_df['cb_person_default_on_file'] == 'Y').astype(int)
ew_df['risky_grade'] = (ew_df['loan_grade'].isin(['D','E','F','G'])).astype(int)
ew_df['n_flags'] = ew_df[['high_dti','high_lpi','high_rate','renter','prior_default','risky_grade']].sum(axis=1)

flag_analysis = ew_df.groupby('n_flags')['loan_status'].agg(['mean','count']).reset_index()
flag_analysis.columns = ['n_flags','default_rate','count']
print("\n── Default Rate by Red Flag Count ──")
print(flag_analysis.to_string(index=False))

# Fairness Analysis
print("""
Fairness is something Golec & AlabdulJalil (2025) flagged as an open
research gap in ML credit models. So I wanted to check our data explicitly.

Looking at the IV analysis:
  Gender:         IV = 0.000, zero predictive power
  Marital status: IV = 0.000, zero predictive power
  Education:      IV = 0.000, zero predictive power
  Country:        default rates are basically identical across US/UK/Canada

This is actually good news. We can safely exclude all of these from the
model without losing any discrimination power, and it ensures compliance
with anti-discrimination regulations.
""")

# Verify fairness claims
print("── Default Rate by Protected Attributes ──")
for col in ['gender', 'education_level', 'marital_status', 'country']:
    print(f"\n{col}:")
    print(df.groupby(col)['loan_status'].agg(['mean','count']).round(4).to_string())

# Early warning chart
fig, ax = plt.subplots(figsize=(10, 6))
portfolio_dr = ew_df['loan_status'].mean()
colors_rf = [COLORS['good'] if dr < 0.20 else COLORS['warn'] if dr < 0.40 else COLORS['accent']
             for dr in flag_analysis['default_rate']]
bars = ax.bar(flag_analysis['n_flags'], flag_analysis['default_rate']*100, color=colors_rf, edgecolor='white')
ax.axhline(y=portfolio_dr*100, color=COLORS['dark'], linestyle='--', alpha=0.5, label=f'Portfolio Avg ({portfolio_dr:.1%})')
ax.set_xlabel('Number of Red Flags'); ax.set_ylabel('Default Rate (%)')
ax.set_title('Early Warning: Default Rate by Red Flag Count', fontsize=14, fontweight='bold')
ax.legend()
for bar, dr, cnt in zip(bars, flag_analysis['default_rate'], flag_analysis['count']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{dr:.0%}\n(n={cnt:,})', ha='center', fontsize=8, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig11_early_warning.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: fig11_early_warning.png")


SECTION 10: EARLY WARNING & FAIRNESS ASSESSMENT

── Default Rate by Red Flag Count ──
 n_flags  default_rate  count
       0      0.052304   9005
       1      0.088238  10925
       2      0.198404   5766
       3      0.682973   3700
       4      0.707947   1510
       5      0.808652    601
       6      1.000000    164

Fairness is something Golec & AlabdulJalil (2025) flagged as an open
research gap in ML credit models. So I wanted to check our data explicitly.

Looking at the IV analysis:
  Gender:         IV = 0.000, zero predictive power
  Marital status: IV = 0.000, zero predictive power
  Education:      IV = 0.000, zero predictive power
  Country:        default rates are basically identical across US/UK/Canada

This is actually good news. We can safely exclude all of these from the
model without losing any discrimination power, and it ensures compliance
with anti-discrimination regulations.

── Default Rate by Protected Attributes ──

gender:
          mean  count
gender 

## 11. Cross-validation

In [21]:
print("\n" + "=" * 80)
print("SECTION 11: CROSS-VALIDATION")
print("=" * 80)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_full_woe = pd.concat([X_train_woe, X_test_woe])
y_full = pd.concat([y_train, y_test])

lr_cv = cross_val_score(
    LogisticRegression(penalty='l2', C=1.0, solver='lbfgs', max_iter=1000, random_state=42),
    X_full_woe, y_full, cv=cv, scoring='roc_auc')
print(f"\nLR 5-Fold CV AUC: {lr_cv.mean():.4f} ± {lr_cv.std():.4f}")
print(f"CV Stability (CoV): {lr_cv.std()/lr_cv.mean():.4f}")


SECTION 11: CROSS-VALIDATION

LR 5-Fold CV AUC: 0.8751 ± 0.0057
CV Stability (CoV): 0.0065


## 12. A/B model comparison
Formal benchmarking of the traditional scorecard vs the ML model. The Gini gap quantifies the cost of interpretability.

In [22]:
print("\n" + "=" * 80)
print("SECTION 12: A/B MODEL COMPARISON: TRADITIONAL vs ML BENCHMARK")
print("=" * 80)
print("""
This section formalizes the comparison between the two approaches.
In production, the CRM team builds multiple candidate models and presents
a recommendation based on discrimination, stability, interpretability,
and regulatory acceptability. This is standard model governance practice.
""")

ab_comparison = pd.DataFrame({
    'Metric': ['Gini (test)', 'KS (test)', 'Gini gap (train-test)', 'PSI',
               'Brier score', 'CV AUC mean', 'Interpretable?', 'Regulatory accepted?',
               'Recommendation'],
    'LR Scorecard': [
        f"{metrics['Logistic Regression']['Gini_test']:.1%}",
        f"{metrics['Logistic Regression']['KS_test']:.1%}",
        f"{metrics['Logistic Regression']['Gini_train'] - metrics['Logistic Regression']['Gini_test']:.4f}",
        f"{metrics['Logistic Regression']['PSI']:.4f}",
        f"{metrics['Logistic Regression']['Brier_test']:.4f}",
        f"{lr_cv.mean():.4f}",
        'Yes: coefficients map to scorecard points',
        'Yes: standard for Basel III IRB',
        'PRODUCTION MODEL'
    ],
    'XGBoost + SHAP': [
        f"{metrics['XGBoost']['Gini_test']:.1%}",
        f"{metrics['XGBoost']['KS_test']:.1%}",
        f"{metrics['XGBoost']['Gini_train'] - metrics['XGBoost']['Gini_test']:.4f}",
        f"{metrics['XGBoost']['PSI']:.4f}",
        f"{metrics['XGBoost']['Brier_test']:.4f}",
        'N/A',
        'Partial: SHAP provides post-hoc explanations',
        'Emerging: requires model governance approval',
        'BENCHMARK ONLY'
    ],
})
print(ab_comparison.to_string(index=False))

gini_gap = metrics['XGBoost']['Gini_test'] - metrics['Logistic Regression']['Gini_test']
print(f"""
── A/B Comparison Conclusion ──

Gini difference: {gini_gap:.1%} (XGBoost higher)
Stability:       LR is more stable ({metrics['Logistic Regression']['Gini_train'] - metrics['Logistic Regression']['Gini_test']:.1%} gap vs {metrics['XGBoost']['Gini_train'] - metrics['XGBoost']['Gini_test']:.1%} for XGBoost)

The {gini_gap:.0%}-point Gini gap is the "cost of interpretability": what we
trade for regulatory compliance and auditability. SHAP analysis confirms both
models agree on the top risk drivers (loan grade, LTI, income, home ownership),
which validates the LR scorecard's feature selection.

RECOMMENDATION: Deploy LR scorecard for production. Maintain XGBoost as a
monitoring benchmark: if the LR Gini degrades but XGBoost holds, it signals
the LR model's functional form (linear on WoE) is becoming inadequate and
redevelopment should consider non-linear approaches.
""")


SECTION 12: A/B MODEL COMPARISON: TRADITIONAL vs ML BENCHMARK

This section formalizes the comparison between the two approaches.
In production, the CRM team builds multiple candidate models and presents
a recommendation based on discrimination, stability, interpretability,
and regulatory acceptability. This is standard model governance practice.

               Metric                              LR Scorecard                               XGBoost + SHAP
          Gini (test)                                     73.7%                                        86.4%
            KS (test)                                     59.7%                                        71.7%
Gini gap (train-test)                                    0.0198                                       0.0459
                  PSI                                    0.0141                                       0.0014
          Brier score                                    0.1004                                       0.

## 13. Production monitoring framework
Simulating 12 months of post-deployment monitoring: monthly Gini/PSI tracking and feature-level drift detection. This is what the CRM team does after deployment.

In [23]:
print("\n" + "=" * 80)
print("SECTION 13: PRODUCTION MONITORING FRAMEWORK")
print("=" * 80)
print("""
Building the model is only half the job. In production, the CRM team
monitors the scorecard continuously to catch degradation early. I wanted
to simulate what that looks like because the JD specifically mentions
"monitor, back-test and report performance of the models."

Typical monitoring cadence for a retail scorecard:
  Monthly:   PSI on overall score and key input features, Gini/KS tracking
  Quarterly: calibration review (does predicted PD still match observed DR?)
  Annually:  full back-test, vintage analysis, redevelopment assessment
""")

# ── 13.1 Simulate monthly monitoring over 12 months ──────────────────────────
# We simulate population drift by gradually shifting the test set
print("── 13.1 Simulated Monthly Gini & PSI Tracking ──")

np.random.seed(42)
n_months = 12
monthly_gini = []
monthly_psi = []
monthly_labels = []

# Development baseline
base_preds = y_test_pred_lr.copy()
base_actuals = y_test.values.copy()

for month in range(1, n_months + 1):
    # Simulate gradual population drift: add noise that increases over time
    drift_factor = month * 0.015  # Small cumulative drift
    noise = np.random.normal(0, drift_factor, size=len(base_preds))
    drifted_preds = np.clip(base_preds + noise, 0.001, 0.999)

    # Simulate slight increase in actual default rate over time (economic cycle)
    flip_prob = month * 0.003
    drifted_actuals = base_actuals.copy()
    flip_mask = np.random.random(len(drifted_actuals)) < flip_prob
    drifted_actuals[flip_mask & (drifted_actuals == 0)] = 1

    # Compute metrics
    try:
        g = gini_coefficient(drifted_actuals, drifted_preds)
    except:
        g = monthly_gini[-1] if monthly_gini else 0.70
    p = calculate_psi(base_preds, drifted_preds)

    monthly_gini.append(g)
    monthly_psi.append(p)
    monthly_labels.append(f'M{month}')

print(f"\n{'Month':<8} {'Gini':>8} {'PSI':>10} {'Status':>20}")
print("-" * 50)
for i in range(n_months):
    gini_status = "OK" if monthly_gini[i] > metrics['Logistic Regression']['Gini_test'] * 0.90 else "INVESTIGATE"
    psi_status = "OK" if monthly_psi[i] < 0.10 else ("INVESTIGATE" if monthly_psi[i] < 0.25 else "REBUILD")
    status = "OK" if gini_status == "OK" and psi_status == "OK" else f"Gini:{gini_status} PSI:{psi_status}"
    print(f"M{i+1:<7} {monthly_gini[i]:>7.1%} {monthly_psi[i]:>10.4f} {status:>20}")

# ── 13.2 Feature-level PSI monitoring ─────────────────────────────────────────
print("\n── 13.2 Feature-Level PSI (checking which inputs are drifting) ──")

key_features_monitor = ['loan_int_rate', 'loan_percent_income', 'person_income',
                         'debt_to_income_ratio', 'loan_amnt', 'person_emp_length']

# Simulate by splitting test set into first half vs second half (proxy for time)
mid = len(X_test) // 2
feature_psi_results = []
for feat in key_features_monitor:
    try:
        feat_psi = calculate_psi(
            X_test[feat].values[:mid].astype(float),
            X_test[feat].values[mid:].astype(float),
            bins=8
        )
        status = "OK" if feat_psi < 0.10 else ("INVESTIGATE" if feat_psi < 0.25 else "REBUILD")
        feature_psi_results.append({'Feature': feat, 'PSI': feat_psi, 'Status': status})
    except:
        feature_psi_results.append({'Feature': feat, 'PSI': 0.0, 'Status': 'OK'})

feat_psi_df = pd.DataFrame(feature_psi_results)
print(feat_psi_df.to_string(index=False))
print("""
Feature-level PSI catches problems BEFORE the overall score PSI triggers.
If a single feature drifts (e.g., applicant income distribution shifts),
the CRM team investigates the cause before the model's overall discrimination
degrades. This is proactive monitoring.
""")

# ── 13.3 Monitoring dashboard visualization ───────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Gini over time
ax = axes[0, 0]
dev_gini = metrics['Logistic Regression']['Gini_test']
ax.plot(range(n_months), [g * 100 for g in monthly_gini], 'o-', color=COLORS['primary'], linewidth=2, markersize=6)
ax.axhline(y=dev_gini * 100, color=COLORS['good'], linestyle='--', linewidth=1.5, label=f'Development Gini ({dev_gini:.1%})')
ax.axhline(y=dev_gini * 100 * 0.90, color=COLORS['accent'], linestyle='--', linewidth=1.5, label=f'Alert threshold (90% of dev)')
ax.fill_between(range(n_months), dev_gini * 100 * 0.90, 0, alpha=0.05, color=COLORS['accent'])
ax.set_xticks(range(n_months))
ax.set_xticklabels(monthly_labels, fontsize=8)
ax.set_ylabel('Gini (%)')
ax.set_title('Monthly Gini tracking', fontweight='bold')
ax.legend(fontsize=8)
ax.set_ylim(50, 85)

# PSI over time
ax = axes[0, 1]
ax.bar(range(n_months), monthly_psi,
       color=[COLORS['good'] if p < 0.10 else COLORS['warn'] if p < 0.25 else COLORS['accent'] for p in monthly_psi],
       edgecolor='white')
ax.axhline(y=0.10, color=COLORS['warn'], linestyle='--', linewidth=1.5, label='Investigate (0.10)')
ax.axhline(y=0.25, color=COLORS['accent'], linestyle='--', linewidth=1.5, label='Rebuild (0.25)')
ax.set_xticks(range(n_months))
ax.set_xticklabels(monthly_labels, fontsize=8)
ax.set_ylabel('PSI')
ax.set_title('Monthly PSI tracking', fontweight='bold')
ax.legend(fontsize=8)

# Calibration by score band (rating table)
ax = axes[1, 0]
test_result = pd.DataFrame({
    'score': test_scores,
    'actual_default': y_test.values,
    'predicted_pd': y_test_pred_lr,
})
test_result['score_band'] = pd.qcut(test_result['score'], q=10, duplicates='drop')
rating = test_result.groupby('score_band', observed=True).agg(
    observed_dr=('actual_default', 'mean'),
    predicted_pd=('predicted_pd', 'mean'),
    count=('actual_default', 'count'),
).reset_index()
x_pos = range(len(rating))
ax.bar(x_pos, rating['observed_dr'] * 100,
       color=[COLORS['good'] if dr < 0.15 else COLORS['warn'] if dr < 0.35 else COLORS['accent']
              for dr in rating['observed_dr']], edgecolor='white', alpha=0.7, label='Observed DR')
ax.plot(x_pos, rating['predicted_pd'] * 100, 'ko-', markersize=5, label='Predicted PD')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'Band {i+1}' for i in x_pos], rotation=45, fontsize=7)
ax.set_ylabel('Default Rate (%)')
ax.set_title('Calibration by score band (back-test)', fontweight='bold')
ax.legend(fontsize=8)

# Feature-level PSI
ax = axes[1, 1]
feat_colors = [COLORS['good'] if row['PSI'] < 0.10 else COLORS['warn'] if row['PSI'] < 0.25
               else COLORS['accent'] for _, row in feat_psi_df.iterrows()]
bars = ax.barh(range(len(feat_psi_df)), feat_psi_df['PSI'], color=feat_colors, edgecolor='white')
ax.set_yticks(range(len(feat_psi_df)))
ax.set_yticklabels(feat_psi_df['Feature'], fontsize=9)
ax.axvline(x=0.10, color=COLORS['warn'], linestyle='--', linewidth=1.5, label='Investigate')
ax.axvline(x=0.25, color=COLORS['accent'], linestyle='--', linewidth=1.5, label='Rebuild')
ax.set_xlabel('PSI')
ax.set_title('Feature-level PSI (input stability)', fontweight='bold')
ax.legend(fontsize=8)
ax.invert_yaxis()
for bar, val in zip(bars, feat_psi_df['PSI']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=8)

plt.suptitle('Production Monitoring Dashboard: Scorecard Health', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig12_monitoring_dashboard.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: fig12_monitoring_dashboard.png")

print("""
── Monitoring Decision Framework ──

  Monthly check:
    1. Gini within 90% of development?  → If no, trigger review
    2. Score PSI < 0.10?                → If no, investigate population shift
    3. Feature PSI < 0.10 for all?      → If any breach, root-cause analysis

  Quarterly check:
    4. Calibration aligned?             → If predicted ≠ observed, recalibrate
    5. Vintage performance stable?      → If newer cohorts worse, flag early

  Annual check:
    6. Full model back-test             → Comprehensive validation report
    7. Redevelopment assessment          → Is a new model needed?

  This framework maps directly to the JD responsibility:
  "Monitor, back-test and report performance of the models to ensure
   adherence to performance standards and early detection of weaknesses."
""")
print("\n" + "=" * 80)
print("SECTION 14: EXECUTIVE SUMMARY")
print("=" * 80)

print(f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║            CREDIT RISK SCORECARD: EXECUTIVE SUMMARY                       ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║  MODEL PERFORMANCE                                                         ║
║  • LR Scorecard:  Gini = {metrics['Logistic Regression']['Gini_test']:.1%} | KS = {metrics['Logistic Regression']['KS_test']:.1%} | PSI = {metrics['Logistic Regression']['PSI']:.4f}     ║
║  • XGBoost + SHAP: Gini = {metrics['XGBoost']['Gini_test']:.1%} | KS = {metrics['XGBoost']['KS_test']:.1%} | PSI = {metrics['XGBoost']['PSI']:.4f}    ║
║  • LR Scorecard is STABLE (2% Gini gap): recommended for production      ║
║  • XGBoost SHAP validates LR feature selection and adds local explain.     ║
║                                                                            ║
║  REGULATORY CAPITAL (Basel III IRB: ASRF)                                 ║
║  • Total RWA: ${test_capital['rwa_amount'].sum()/1e6:.1f}M on ${test_capital['loan_amnt'].sum()/1e6:.1f}M exposure              ║
║  • Avg Capital Requirement: {test_capital['capital_amount'].sum()/test_capital['loan_amnt'].sum()*100:.1f}%                                        ║
║  • Capital ranges from <2% (AAA) to >20% (high-risk bands)                ║
║                                                                            ║
║  IFRS 9 ECL                                                                ║
║  • Total ECL: ${stage_summary['total_ecl'].sum()/1e6:.1f}M ({stage_summary['total_ecl'].sum()/stage_summary['total_exposure'].sum():.1%} of exposure)                          ║
║  • Stage 2 (Watch) = {stage_summary.loc[stage_summary['stage']=='Stage 2','pct_portfolio'].values[0] if 'Stage 2' in stage_summary['stage'].values else 'N/A'}% of portfolio → proactive monitoring needed  ║
║                                                                            ║
║  STRESS TESTING                                                            ║
║  • Severe stress: ECL increases ~{float(stress_results[2]['Total ECL ($M)'])/float(stress_results[0]['Total ECL ($M)']):.1f}x from baseline                           ║
║  • Portfolio can absorb mild downturn; severe stress requires action       ║
║                                                                            ║
║  FAIRNESS                                                                  ║
║  • Gender, education, marital status: IV ≈ 0 → excluded from model        ║
║  • No geographic bias (US/UK/Canada default rates identical)               ║
║                                                                            ║
║  REFERENCES                                                                ║
║  [1] Basel Committee (1999). Credit Risk Modelling                         ║
║  [2] Noguer i Alonso & Sun (2025). Credit Risk Modeling. SSRN.            ║
║  [3] Golec & AlabdulJalil (2025). Interpretable LLMs. arXiv.              ║
║  [4] Hlongwane et al. (2024). SHAP Scorecards. PLoS ONE.                  ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

# Save outputs
metrics_df.to_csv(f'{OUTPUT_DIR}/model_metrics.csv')
stress_df.to_csv(f'{OUTPUT_DIR}/stress_test_results.csv', index=False)
stage_summary.to_csv(f'{OUTPUT_DIR}/ifrs9_staging.csv', index=False)
cap_by_band.to_csv(f'{OUTPUT_DIR}/basel_capital_by_band.csv', index=False)
iv_df.to_csv(f'{OUTPUT_DIR}/information_value.csv')

print("\n✓ All outputs saved.")
print("=" * 80)


SECTION 13: PRODUCTION MONITORING FRAMEWORK

Building the model is only half the job. In production, the CRM team
monitors the scorecard continuously to catch degradation early. I wanted
to simulate what that looks like because the JD specifically mentions
"monitor, back-test and report performance of the models."

Typical monitoring cadence for a retail scorecard:
  Monthly:   PSI on overall score and key input features, Gini/KS tracking
  Quarterly: calibration review (does predicted PD still match observed DR?)
  Annually:  full back-test, vintage analysis, redevelopment assessment

── 13.1 Simulated Monthly Gini & PSI Tracking ──

Month        Gini        PSI               Status
--------------------------------------------------
M1         73.0%     0.0173                   OK
M2         70.8%     0.0487                   OK
M3         69.8%     0.1019 Gini:OK PSI:INVESTIGATE
M4         67.8%     0.1619 Gini:OK PSI:INVESTIGATE
M5         66.8%     0.2075 Gini:OK PSI:INVESTIGATE
M